**Lab 8: Check the claims**

In [ ]:
# Importing the relevand libraries.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Applying Seaborn's darkgrid theme.
sns.set_theme(style='darkgrid')

# Creating a python list containing column names with dates for parsing.
date_columns = ['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review']

# Loading the CSV and parsing the date_columns for the proper datatype in the DataFrame.
listings = pd.read_csv('data/listings_clean.csv', parse_dates=date_columns)

# Displaying the number of rows and columns in the DataFrame.
print(listings.shape)

**Claim 1: "A night in Stockholm costs 1,791."**

In [ ]:
price = listings['price']

# print(column.mean(), column.median())

price.nunique()

In [ ]:
# print("None missing values for 'price':", column.notna().sum())
price.describe()

In [ ]:
print("Missing values for price (in %):", (price.isna().mean() * 100).round(1))
print("Values with a price (in %):", (price.notna().mean() * 100).round(1))

In [ ]:
# Examining possible correlation between values for price and availability (open days).
pd.crosstab(
    listings['price'].isna(),
    listings['availability_365'] == 0,
    rownames=['Price is missing'],
    colnames=['No open days']
) # Of the 1,617 listings with missing prices, 1,490 also have zero open days. 127 missing-price listings have at least one open day.

In [ ]:
annual_zero = listings[(listings['availability_365'] == 0)]

print(len(annual_zero))

In [ ]:
missing_price = listings['price'].isna()
no_open_days = listings['availability_365'] == 0

print(((missing_price & no_open_days).sum() / missing_price.sum() * 100).round(1))

In [ ]:
# Compare long-stay and other listings.
long_stay = listings['minimum_nights'] >= 28

print(
    listings
    .groupby(long_stay)['price']
    .agg(['count', 'median'])
    .round(1)
)

In [ ]:
# Checking when the data was collected.
print(
    listings['last_scraped']
    .value_counts()
    .sort_index()
)

Verdict on claim 1:  Holds with reservations.
**Verdict: Holds with reservations.**
On 26 and 27 September 2026, the median price per night was 1,791 SEK among the 2,957 Airbnb listings in Stockholm that had a recorded price, representing approximately 65% of all listings in the dataset.
But this is based on only 2,957 of the 4,574 Airbnb listings in Stockholm. The remaining 1,617 listings, or 35.4%, have no recorded price.
The claim describes priced Airbnb listings in the dataset, not the cost of accommodation throughout Stockholm.



___

**Claim 2: "The listings with a perfect 5.00 rating are the best in Stockholm."**

In [ ]:
mask = listings['has_reviews']
rated = listings[mask] # The remaining 644 listings have no reviews and no rating.
perfect = rated['review_scores_rating'] == 5.00

print("Rated listings:", len(rated))
print("Listings with a perfect score:", perfect.sum())
print('Percentage of rated listings:', (perfect.mean() * 100).round(1))

In [ ]:
rating_group = perfect.map({
    True: 'Perfect rating',
    False: 'Other rating'
})

rated.groupby(rating_group)['number_of_reviews'].agg(
    ['count', 'median']
)

In [ ]:
perfect_with_20 = (
    perfect
    & (rated['number_of_reviews'] >= 20)
)

print(
    'Perfect listings with at least 20 reviews:',
    perfect_with_20.sum()
)

print(
    'Share of perfect listings with at least 20 reviews:',
    (perfect_with_20.sum() / perfect.sum() * 100).round(1)
)

Verdict: Does not hold.
Among rated Airbnb listings in the dataset, 38% have a perfect 5.00 rating, but 91% of those listings have fewer than 20 reviews, so the rating alone provides little evidence that they are the best listings.

___

**Claim 3: "The market has grown: only 131 listings started in 2016, but 610 in 2024."**